In [14]:
# ==============================================================================
# Cal Regression: ALL-IN-ONE caret pipeline
# - 12 files × 4 targets × multiple models
# - NA complete-case
# - Feature cleaning: zero-variance 제거 + near-duplicate correlation 제거
# - Splits: OOD (centroid distance) + CV10 (OOF) + Train fit
# - Metrics: R2/RMSE/MAE/RPD for Train, CV10(OOF), OOD
# - Saves: model .rds, prediction .csv, summary .csv, importance .csv, overfit .csv
# - Fixes: OOF pred binding mismatch, PLS VIP robust guard
# - ★本版の仕様:
#   1. 4つの目的変数(Jsc, Voc, FF, PCEmax)は **すべて説明変数から除外**
#   2. Train / CV10_OOF / OOD の3種の指標を出力
#   3. 予測CSVは Train / CV10_OOF / OOD_Test を同じ列構造で rbind
#   4. Train vs CV10_OOF の差から Overfit_Flag / Overfit_Label / Delta_* を算出し保存
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(kernlab)  # SVM / GPR
  library(earth)    # gcvEarth
  library(pls)      # PLS
})

set.seed(42)

# -------------------------------
# USER SETTINGS
# -------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
today <- format(Sys.Date(), "%Y%m%d")

file_names <- c(
  "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
  "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
  "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
  "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
)

# 実行するモデル
models_to_run <- c(
  "SVM_Linear"
  # "SVM_Radial"
  # "GPR_Radial"
  # "GPR_Linear"
  # "gcvEarth"
#   "PLS"
#   "kNN"
  # "PPR"
)

# OOD ratio
ood_ratio <- 0.10

# Overfit rule thresholds
overfit_delta_r2_thr   <- 0.10
overfit_delta_rmse_rel <- 0.10  # 10% of Train_RMSE

# -------------------------------
# HELPERS
# -------------------------------

safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  return(as.numeric(r^2))
}

safe_rmse <- function(y, p) rmse(y, p)
safe_mae  <- function(y, p) mae(y, p)

safe_rpd <- function(y, rmse_val) {
  if (is.na(rmse_val) || rmse_val == 0) return(NA_real_)
  return(as.numeric(sd(y) / rmse_val))
}

create_ood_split <- function(df, feature_cols, ood_ratio = 0.1) {
  X <- df[, feature_cols, drop = FALSE]
  centroid <- colMeans(X)
  dists <- apply(X, 1, function(row) sqrt(sum((row - centroid)^2)))
  num_ood <- max(1, floor(nrow(df) * ood_ratio))
  ood_indices <- order(dists, decreasing = TRUE)[1:num_ood]
  list(
    train = df[-ood_indices, , drop = FALSE],
    test  = df[ ood_indices, , drop = FALSE]
  )
}

scale_neg1_to_1 <- function(train_df, test_df, feature_cols) {
  pp <- preProcess(train_df[, feature_cols, drop = FALSE], method = "range")
  train_01 <- predict(pp, train_df[, feature_cols, drop = FALSE])
  test_01  <- predict(pp, test_df[, feature_cols, drop = FALSE])
  train_df[, feature_cols] <- train_01 * 2 - 1
  test_df[, feature_cols]  <- test_01 * 2 - 1
  list(train = train_df, test = test_df)
}

# --- OOF extraction (bestTune only) + aggregate to one rowIndex
extract_oof_best <- function(model, train_scaled) {
  pred_oof <- model$pred
  if (is.null(pred_oof) || nrow(pred_oof) == 0) return(NULL)

  # filter to bestTune rows
  for (pname in names(model$bestTune)) {
    pred_oof <- pred_oof[pred_oof[[pname]] == model$bestTune[[pname]], , drop = FALSE]
  }
  if (nrow(pred_oof) == 0) return(NULL)

  # aggregate (in case multiple predictions exist per rowIndex)
  pred_oof_agg <- pred_oof %>%
    dplyr::group_by(rowIndex) %>%
    dplyr::summarise(
      Predicted = mean(pred, na.rm = TRUE),
      Observed  = mean(obs,  na.rm = TRUE),
      .groups = "drop"
    )

  pred_oof_agg$SampleID <- rownames(train_scaled)[pred_oof_agg$rowIndex]
  df_oof_out <- pred_oof_agg %>%
    dplyr::select(SampleID, Observed, Predicted) %>%
    dplyr::mutate(Type = "CV10_OOF")
  df_oof_out
}

# --- Permutation Importance (simple; uses Train fit R2 as baseline)
calc_permutation_importance <- function(model, data_with_target, target_col, feature_cols, base_r2) {
  out <- list()
  for (col in feature_cols) {
    temp <- data_with_target
    temp[[col]] <- sample(temp[[col]])
    preds <- tryCatch(
      predict(model, temp[, feature_cols, drop = FALSE]),
      error = function(e) NA
    )
    if (all(is.na(preds))) {
      out[[col]] <- NA_real_
      next
    }
    new_r2 <- safe_r2(temp[[target_col]], preds)
    if (is.na(new_r2)) new_r2 <- 0
    out[[col]] <- base_r2 - new_r2
  }
  out
}

# --- PLS VIP (robust; avoids subscript out of bounds)
calc_vip_pls <- function(pls_mvr, ncomp) {
  if (is.null(pls_mvr$loading.weights) || is.null(pls_mvr$scores)) return(NULL)

  W <- as.matrix(pls_mvr$loading.weights)
  Tt <- as.matrix(pls_mvr$scores)
  Q  <- as.matrix(pls_mvr$Yloadings)

  A <- min(ncomp, ncol(W), ncol(Tt), ncol(Q))
  if (A < 1) return(NULL)

  W <- W[, 1:A, drop = FALSE]
  Tt <- Tt[, 1:A, drop = FALSE]
  Q  <- Q[, 1:A, drop = FALSE]

  p <- nrow(W)
  SS <- numeric(A)
  for (a in 1:A) {
    SS[a] <- sum(Tt[, a]^2) * sum(Q[, a]^2)
  }
  SSY <- sum(SS)
  if (!is.finite(SSY) || SSY == 0) return(NULL)

  Wnorm2 <- colSums(W^2)
  vip <- numeric(p)

  for (j in 1:p) {
    wsum <- 0
    for (a in 1:A) {
      if (Wnorm2[a] == 0) next
      wsum <- wsum + SS[a] * (W[j, a]^2 / Wnorm2[a])
    }
    vip[j] <- sqrt(p * wsum / SSY)
  }
  names(vip) <- rownames(W)
  vip
}

# --- model registry (caret method + tuning grid)
get_model_spec <- function(model_key) {
  if (model_key == "SVM_Linear") {
    list(
      caret_method = "svmLinear",
      model_name   = "SVM_Linear",
      tune_grid    = expand.grid(C = c(0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100)),
      best_params  = function(m) paste0("C=", m$bestTune$C)
    )
  } else if (model_key == "SVM_Radial") {
    list(
      caret_method = "svmRadial",
      model_name   = "SVM_Radial",
      tune_grid    = expand.grid(sigma = 2^seq(-15, 3, length = 7), C = c(1, 10, 100)),
      best_params  = function(m) paste0("sigma=", round(m$bestTune$sigma, 6), ", C=", m$bestTune$C)
    )
  } else if (model_key == "GPR_Radial") {
    list(
      caret_method = "gaussprRadial",
      model_name   = "GPR_Radial",
      tune_grid    = expand.grid(sigma = 2^seq(-15, 3, length = 7)),
      best_params  = function(m) paste0("sigma=", m$bestTune$sigma)
    )
  } else if (model_key == "GPR_Linear") {
    list(
      caret_method = "gaussprLinear",
      model_name   = "GPR_Linear",
      # ★注意：caret のパラメータ名は要確認（ここでは仮に 'sigma' を使用）
      tune_grid    = expand.grid(sigma = 2^seq(-15, 3, length = 7)),
      best_params  = function(m) paste0("sigma=", m$bestTune$sigma)
    )
  } else if (model_key == "gcvEarth") {
    list(
      caret_method = "gcvEarth",
      model_name   = "gcvEarth",
      tune_grid    = expand.grid(degree = 1:3),
      best_params  = function(m) paste0("degree=", m$bestTune$degree)
    )
  } else if (model_key == "PLS") {
    list(
      caret_method = "pls",
      model_name   = "PLS",
      tune_grid_fn = function(max_ncomp) expand.grid(ncomp = 1:max_ncomp),
      best_params  = function(m) paste0("ncomp=", m$bestTune$ncomp)
    )
  } else if (model_key == "kNN") {
    list(
      caret_method = "knn",
      model_name   = "kNN",
      tune_grid    = expand.grid(k = seq(1, 21, by = 2)),
      best_params  = function(m) paste0("k=", m$bestTune$k)
    )
  } else if (model_key == "PPR") {
    list(
      caret_method = "ppr",
      model_name   = "PPR",
      tune_grid    = expand.grid(nterms = 1:10),
      best_params  = function(m) paste0("nterms=", m$bestTune$nterms)
    )
  } else {
    stop("Unknown model_key: ", model_key)
  }
}

# -------------------------------
# MAIN RUNNER (per model)
# -------------------------------
run_one_model <- function(model_key) {

  spec <- get_model_spec(model_key)
  model_name <- spec$model_name

  out_dir <- file.path(getwd(), paste0(today, "_Rebuilt_12Files_", model_name, "_Train_CV10OOF_OOD"))
  if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

  cat("\n======================================================\n")
  cat(sprintf("--- %s Analysis Started: %s ---\n", model_name, today))
  cat(sprintf("[INFO] Input base path: %s\n", base_path))
  cat(sprintf("[INFO] Output dir     : %s\n", out_dir))
  cat("======================================================\n")

  summary_data    <- dplyr::tibble()
  importance_data <- dplyr::tibble()

  for (file in file_names) {
    filepath <- file.path(base_path, file)
    if (!file.exists(filepath)) {
      cat(sprintf("Skipping: %s (Not found)\n", file))
      next
    }

    cat(sprintf("\n=== Processing: %s ===\n", file))

    df_raw <- tryCatch(
      read.csv(filepath, row.names = 1, check.names = FALSE),
      error = function(e) NULL
    )
    if (is.null(df_raw)) {
      cat("  [ERROR] Failed to read file.\n")
      next
    }
    if ("X" %in% colnames(df_raw)) df_raw$X <- NULL

    for (target in target_vars) {
      if (!target %in% colnames(df_raw)) next

      # numeric only + complete-case
      numeric_cols <- sapply(df_raw, is.numeric)
      df_temp <- df_raw[, numeric_cols, drop = FALSE] %>% na.omit()
      if (nrow(df_temp) < 20) next

      # --------------------------------------------------
      # ★変更ポイント: 特徴量 = 数値列 から 4目的変数すべてを除外
      #   → Jscモデルの場合でも Voc/FF/PCEmax は説明変数に使わない
      # --------------------------------------------------
      features <- setdiff(colnames(df_temp), target_vars)
      if (length(features) == 0) next

      # remove zero variance
      vars <- sapply(df_temp[, features, drop = FALSE], var, na.rm = TRUE)
      features <- names(vars)[vars > 0 & !is.na(vars)]
      if (length(features) == 0) next

      # remove near-duplicate features (very high corr)
      if (length(features) > 1) {
        cor_mat <- suppressWarnings(
          cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
        )
        cor_mat[is.na(cor_mat)] <- 0
        high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
        if (length(high_corr) > 0) features <- features[-high_corr]
      }
      if (length(features) == 0) next

      # OOD split (before scaling; scaling fit on train only)
      split_res  <- create_ood_split(df_temp, features, ood_ratio = ood_ratio)
      train_set  <- split_res$train
      test_set   <- split_res$test

      # scaling (-1..1) for ALL models (consistent)
      scaled_res   <- scale_neg1_to_1(train_set, test_set, features)
      train_scaled <- scaled_res$train
      test_scaled  <- scaled_res$test

      # trainControl
      train_ctrl <- trainControl(
        method = "cv",
        number = 10,
        savePredictions = "final"
      )

      # tune grid
      if (model_key == "PLS") {
        max_ncomp <- min(10, nrow(train_scaled) - 1, length(features))
        if (max_ncomp < 1) next
        tune_grid <- spec$tune_grid_fn(max_ncomp)
      } else {
        tune_grid <- spec$tune_grid
      }

      # train caret model
      set.seed(42)
      model <- tryCatch(
        train(
          x = train_scaled[, features, drop = FALSE],
          y = train_scaled[[target]],
          method   = spec$caret_method,
          metric   = "RMSE",
          trControl = train_ctrl,
          tuneGrid  = tune_grid
        ),
        error = function(e) {
          cat(sprintf("  [ERROR] train() failed for Target=%s : %s\n", target, e$message))
          NULL
        }
      )
      if (is.null(model)) next

      # Predictions: Train (fit), OOD (hold-out)
      pred_train <- tryCatch(
        predict(model, train_scaled[, features, drop = FALSE]),
        error = function(e) NA
      )
      pred_test <- tryCatch(
        predict(model, test_scaled[, features, drop = FALSE]),
        error = function(e) NA
      )

      # OOF predictions
      df_oof_out <- extract_oof_best(model, train_scaled)
      if (is.null(df_oof_out) || nrow(df_oof_out) < 3) {
        cat(sprintf("  [WARN] OOF extraction failed/too small for Target=%s. Skipping.\n", target))
        next
      }

      # Metrics: Train
      r2_train   <- safe_r2(train_scaled[[target]], pred_train)
      rmse_train <- safe_rmse(train_scaled[[target]], pred_train)
      mae_train  <- safe_mae(train_scaled[[target]], pred_train)
      rpd_train  <- safe_rpd(train_scaled[[target]], rmse_train)

      # Metrics: CV10 OOF
      r2_cv10   <- safe_r2(df_oof_out$Observed, df_oof_out$Predicted)
      rmse_cv10 <- safe_rmse(df_oof_out$Observed, df_oof_out$Predicted)
      mae_cv10  <- safe_mae(df_oof_out$Observed, df_oof_out$Predicted)
      rpd_cv10  <- safe_rpd(df_oof_out$Observed, rmse_cv10)

      # Metrics: OOD
      r2_ood   <- safe_r2(test_scaled[[target]], pred_test)
      rmse_ood <- safe_rmse(test_scaled[[target]], pred_test)
      mae_ood  <- safe_mae(test_scaled[[target]], pred_test)
      rpd_ood  <- safe_rpd(test_scaled[[target]], rmse_ood)

      # Overfit check (Train vs CV10_OOF)
      delta_r2   <- r2_train - r2_cv10
      delta_rmse <- rmse_cv10 - rmse_train
      overfit_flag <- (is.finite(delta_r2) && delta_r2 >= overfit_delta_r2_thr) ||
        (is.finite(delta_rmse) && is.finite(rmse_train) &&
           delta_rmse >= overfit_delta_rmse_rel * abs(rmse_train))
      overfit_label <- ifelse(overfit_flag, "Overfit_suspected", "OK")

      best_params_str <- spec$best_params(model)

      # Save model bundle
      rds_file <- paste0(
        "fixed_", today, "_", model_name, "_model_",
        tools::file_path_sans_ext(file), "_", target, ".rds"
      )
      saveRDS(
        list(
          model        = model,
          training_data = train_scaled,
          ood_test_data = test_scaled,
          features     = features,
          target       = target
        ),
        file = file.path(out_dir, rds_file)
      )

      # ===========================
      # 予測CSV: Train / CV10_OOF / OOD_Test
      # → 完全に同じ列構造: SampleID, Observed, Predicted, Type
      # ===========================
      df_train_out <- data.frame(
        SampleID  = rownames(train_scaled),
        Observed  = train_scaled[[target]],
        Predicted = as.numeric(pred_train),
        Type      = "Train",
        stringsAsFactors = FALSE
      )
      df_ood_out <- data.frame(
        SampleID  = rownames(test_scaled),
        Observed  = test_scaled[[target]],
        Predicted = as.numeric(pred_test),
        Type      = "OOD_Test",
        stringsAsFactors = FALSE
      )
      df_pred_all <- dplyr::bind_rows(df_train_out, df_oof_out, df_ood_out)

      pred_file <- paste0(
        "fixed_", today, "_", model_name, "_",
        tools::file_path_sans_ext(file), "_", target, "_pred.csv"
      )
      write.csv(df_pred_all, file.path(out_dir, pred_file), row.names = FALSE)

      # Importance
      if (model_key == "PLS") {
        vip_vals <- tryCatch(
          calc_vip_pls(model$finalModel, model$bestTune$ncomp),
          error = function(e) NULL
        )
        if (!is.null(vip_vals) && length(vip_vals) > 0) {
          vip_df <- data.frame(
            File            = file,
            Target          = target,
            Feature         = names(vip_vals),
            Importance_Mean = as.numeric(vip_vals),
            Importance_Type = "VIP",
            stringsAsFactors = FALSE
          )
          importance_data <- dplyr::bind_rows(importance_data, vip_df)
        } else {
          cat(sprintf("  [WARN] VIP calculation failed for %s/%s (skipped)\n", file, target))
        }
      } else {
        imps <- calc_permutation_importance(
          model,
          train_scaled,
          target,
          features,
          base_r2 = r2_train
        )
        imp_df <- data.frame(
          File            = file,
          Target          = target,
          Feature         = names(imps),
          Importance_Mean = as.numeric(unlist(imps)),
          Importance_Type = "Permutation_R2drop",
          stringsAsFactors = FALSE
        )
        imp_df <- imp_df %>%
          dplyr::filter(is.finite(Importance_Mean) & abs(Importance_Mean) > 1e-6)
        importance_data <- dplyr::bind_rows(importance_data, imp_df)
      }

      # Summary row (Train / CV10_OOF / OOD)
      summary_row <- data.frame(
        File   = file,
        Target = target,
        Train_R2   = r2_train,
        Train_RMSE = rmse_train,
        Train_MAE  = mae_train,
        Train_RPD  = rpd_train,
        CV10_R2    = r2_cv10,
        CV10_RMSE  = rmse_cv10,
        CV10_MAE   = mae_cv10,
        CV10_RPD   = rpd_cv10,
        OOD_R2     = r2_ood,
        OOD_RMSE   = rmse_ood,
        OOD_MAE    = mae_ood,
        OOD_RPD    = rpd_ood,
        n_samples  = nrow(df_temp),
        n_features = length(features),
        Best_Params = best_params_str,
        Overfit_Flag  = overfit_flag,
        Overfit_Label = overfit_label,
        Delta_R2_TrainMinusCV10   = delta_r2,
        Delta_RMSE_CV10MinusTrain = delta_rmse,
        stringsAsFactors = FALSE
      )
      summary_data <- dplyr::bind_rows(summary_data, summary_row)

      cat(sprintf(
        "  Target=%s | TrainR2=%.3f | CV10R2=%.3f | OODR2=%.3f | %s\n",
        target, r2_train, r2_cv10, r2_ood, overfit_label
      ))
    }
  }

  # Save summary / importance / overfit table
  sum_file <- file.path(out_dir, paste0("fixed_", today, "_", model_name, "_summary.csv"))
  write.csv(summary_data, sum_file, row.names = FALSE)

  imp_file <- file.path(out_dir, paste0("fixed_", today, "_", model_name, "_feature_importance.csv"))
  write.csv(importance_data, imp_file, row.names = FALSE)

  overfit_file <- file.path(out_dir, paste0("fixed_", today, "_", model_name, "_overfit_judgement.csv"))
  overfit_tbl <- summary_data %>%
    dplyr::select(
      File, Target,
      Train_R2, CV10_R2, OOD_R2,
      Train_RMSE, CV10_RMSE, OOD_RMSE,
      Overfit_Flag, Overfit_Label,
      Delta_R2_TrainMinusCV10,
      Delta_RMSE_CV10MinusTrain,
      Best_Params, n_samples, n_features
    )
  write.csv(overfit_tbl, overfit_file, row.names = FALSE)

  cat("\n======================================================\n")
  cat(sprintf("%s Finished.\n", model_name))
  cat(sprintf("Summary    : %s\n", sum_file))
  cat(sprintf("Importance : %s\n", imp_file))
  cat(sprintf("Overfit    : %s\n", overfit_file))
  cat("======================================================\n")
}

# -------------------------------
# RUN ALL SELECTED MODELS
# -------------------------------
for (mk in models_to_run) {
  run_one_model(mk)
}



--- SVM_Linear Analysis Started: 20251218 ---
[INFO] Input base path: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/
[INFO] Output dir     : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/svmLinear/20251218_Rebuilt_12Files_SVM_Linear_Train_CV10OOF_OOD

=== Processing: n_base.csv ===
  Target=Jsc | TrainR2=0.290 | CV10R2=0.258 | OODR2=0.151 | OK
  Target=Voc | TrainR2=0.679 | CV10R2=0.671 | OODR2=0.812 | OK
  Target=FF | TrainR2=0.229 | CV10R2=0.211 | OODR2=0.003 | OK
  Target=PCEmax | TrainR2=0.236 | CV10R2=0.217 | OODR2=0.280 | OK

=== Processing: n_base_OH_rebuilt.csv ===


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: SMILESsnamep1M_3hexylthieno.3.2b.thiopheneP3, SMILESsnamep1M_BDTH3, SMILESsnamep1M_BDTH6, SMILESsnamep1M_BDTH8, SMILESsnamep1M_BDTH9, SMILESsnamep1M_CPDTDTKP, SMILESsnamep1M_PBDTTTST"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) 

  Target=Jsc | TrainR2=0.879 | CV10R2=0.743 | OODR2=0.137 | Overfit_suspected


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: SMILESsnamep1M_3hexylthieno.3.2b.thiopheneP3, SMILESsnamep1M_BDTH3, SMILESsnamep1M_BDTH6, SMILESsnamep1M_BDTH8, SMILESsnamep1M_BDTH9, SMILESsnamep1M_CPDTDTKP, SMILESsnamep1M_PBDTTTST"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) 

  Target=Voc | TrainR2=0.813 | CV10R2=0.707 | OODR2=0.396 | Overfit_suspected


...

  Target=Voc | TrainR2=0.658 | CV10R2=0.541 | OODR2=0.468 | Overfit_suspected


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB, X165.2, X165.5"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' const

  Target=FF | TrainR2=0.600 | CV10R2=0.451 | OODR2=0.387 | Overfit_suspected


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB, X165.2, X165.5"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' const

  Target=PCEmax | TrainR2=0.776 | CV10R2=0.676 | OODR2=0.141 | Overfit_suspected

SVM_Linear Finished.
Summary    : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/svmLinear/20251218_Rebuilt_12Files_SVM_Linear_Train_CV10OOF_OOD/fixed_20251218_SVM_Linear_summary.csv
Importance : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/svmLinear/20251218_Rebuilt_12Files_SVM_Linear_Train_CV10OOF_OOD/fixed_20251218_SVM_Linear_feature_importance.csv
Overfit    : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/svmLinear/20251218_Rebuilt_12Files_SVM_Linear_Train_CV10OOF_OOD/fixed_20251218_SVM_Linear_overfit_judgement.csv


In [7]:
##badpredID

In [8]:
# # ==============================================================================
# # SVM Linear (線形SVM) 解析：完全版
# # (12ファイル対応 + NA完全除去 + モデル・プロットデータ保存 + SampleID対応)
# # ==============================================================================

# library(caret)
# library(kernlab)
# library(dplyr)
# library(Metrics)

# # --- 設定 ---
# set.seed(42)

# # データパス
# base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"

# target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
# today <- format(Sys.Date(), "%Y%m%d")

# # 【新条件】出力先フォルダの設定
# folder_name <- paste0(today, "_Rebuilt_12Files_Results")
# output_path <- file.path("./", folder_name)

# if (!dir.exists(output_path)) {
#   dir.create(output_path, recursive = TRUE)
#   cat(sprintf("Created output directory: %s\n", output_path))
# }

# # 12ファイル対応
# file_names <- c(
#     "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
#     "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
#     "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
#     "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
# )

# # --- 関数定義 ---

# # OOD分割関数
# create_ood_split <- function(df, feature_cols, ood_ratio = 0.1) {
#   X <- df[, feature_cols, drop = FALSE]
#   centroid <- colMeans(X)
#   dists <- apply(X, 1, function(row) sqrt(sum((row - centroid)^2)))
#   num_ood <- max(1, floor(nrow(df) * ood_ratio))
#   ood_indices <- order(dists, decreasing = TRUE)[1:num_ood]
#   return(list(train = df[-ood_indices, ], test = df[ood_indices, ]))
# }

# # スケーリング関数 (-1 to 1)
# scale_neg1_to_1 <- function(train_df, test_df, feature_cols) {
#   pp <- preProcess(train_df[, feature_cols, drop=FALSE], method = "range")
#   train_01 <- predict(pp, train_df[, feature_cols, drop=FALSE])
#   test_01  <- predict(pp, test_df[, feature_cols, drop=FALSE])
#   train_df[, feature_cols] <- train_01 * 2 - 1
#   test_df[, feature_cols]  <- test_01 * 2 - 1
#   return(list(train = train_df, test = test_df))
# }

# # Permutation Importance 計算関数
# calc_permutation_importance <- function(model, data, target_col, feature_cols, base_r2) {
#   importance_list <- list()
#   for (col in feature_cols) {
#     temp_data <- data
#     temp_data[[col]] <- sample(temp_data[[col]])
    
#     # 特徴量のみ渡す
#     preds <- predict(model, temp_data[, feature_cols, drop=FALSE])
    
#     new_r2 <- cor(data[[target_col]], preds)^2
#     if(is.na(new_r2)) new_r2 <- 0
#     importance_list[[col]] <- base_r2 - new_r2
#   }
#   return(importance_list)
# }

# # --- メイン処理 ---
# summary_data <- data.frame()
# feature_importance_data <- data.frame()

# cat(paste0("--- SVM Linear Analysis Started: ", today, " ---\n"))
# cat(paste0("Output: ", output_path, "\n"))

# for (file in file_names) {
#   filepath <- file.path(base_path, file)
#   if (!file.exists(filepath)) {
#     cat(sprintf("Skipping: %s (Not found)\n", file))
#     next
#   }
  
#   cat(sprintf("\n=== Processing SVM Linear: %s ===\n", file))
  
#   tryCatch({
#     df_raw <- read.csv(filepath, row.names = 1, check.names = FALSE)
#   }, error = function(e) {
#     cat(paste("Error reading file:", e$message, "\n"))
#     next
#   })
#   if ("X" %in% colnames(df_raw)) df_raw$X <- NULL
  
#   for (target in target_vars) {
#     if (!target %in% colnames(df_raw)) next
    
#     # 1. データ整理 (NA完全除去)
#     numeric_cols <- sapply(df_raw, is.numeric)
#     df_temp <- df_raw[, numeric_cols, drop=FALSE]
#     df_temp <- na.omit(df_temp)
    
#     if(nrow(df_temp) < 20) next
    
#     features <- setdiff(colnames(df_temp), target)
#     vars <- sapply(df_temp[, features, drop=FALSE], var, na.rm=TRUE)
#     features <- names(vars)[vars > 0 & !is.na(vars)]
    
#     # 重複削除
#     if(length(features) > 1) {
#       cor_mat <- cor(df_temp[, features], use = "pairwise.complete.obs")
#       cor_mat[is.na(cor_mat)] <- 0
#       high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
#       if (length(high_corr) > 0) features <- features[-high_corr]
#     }
#     if(length(features) == 0) next
    
#     # 2. OOD分割
#     split_res <- create_ood_split(df_temp, features, ood_ratio = 0.1)
#     train_set <- split_res$train
#     test_set  <- split_res$test
    
#     # 3. スケーリング (-1 to 1)
#     scaled_res <- scale_neg1_to_1(train_set, test_set, features)
#     train_scaled <- scaled_res$train
#     test_scaled  <- scaled_res$test
    
#     # 4. モデル構築 (SVM Linear)
#     # Cost (C) パラメータの探索
#     tune_grid <- expand.grid(C = c(0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100))
    
#     train_ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")
    
#     set.seed(42)
#     model <- train(
#       x = train_scaled[, features, drop=FALSE],
#       y = train_scaled[[target]],
#       method = "svmLinear",
#       metric = "RMSE",
#       trControl = train_ctrl,
#       tuneGrid = tune_grid
#     )
    
#     # 5. 評価
#     # 【重要】predictには特徴量のみ渡す
#     pred_train <- predict(model, train_scaled[, features, drop=FALSE])
#     r2_train <- cor(train_scaled[[target]], pred_train)^2
#     rmse_train <- rmse(train_scaled[[target]], pred_train)
#     mae_train <- mae(train_scaled[[target]], pred_train)
#     rpd <- sd(train_scaled[[target]]) / rmse_train
    
#     pred_test <- predict(model, test_scaled[, features, drop=FALSE])
#     r2_ood <- cor(test_scaled[[target]], pred_test)^2
#     rmse_ood <- rmse(test_scaled[[target]], pred_test)
    
#     summary_row <- data.frame(
#       File = file, Target = target,
#       R2 = r2_train, RMSE = rmse_train, MAE = mae_train, RPD = rpd,
#       n_samples = nrow(df_temp), n_features = length(features),
#       OOD_R2 = r2_ood, OOD_RMSE = rmse_ood,
#       Best_Params = paste0("C=", model$bestTune$C)
#     )
#     summary_data <- rbind(summary_data, summary_row)

#     # === 保存処理 ===
    
#     # A. モデル本体 (.rds)
#     model_data_bundle <- list(model = model, training_data = train_scaled, ood_test_data = test_scaled)
#     rds_file <- paste0("fixed_", today, "_SVM_Linear_model_", tools::file_path_sans_ext(file), "_", target, ".rds")
#     saveRDS(model_data_bundle, file = file.path(output_path, rds_file))

#     # B. プロット用データ (.csv) - SampleIDを追加
#     df_pred_all <- rbind(
#       data.frame(
#         SampleID = rownames(train_scaled),
#         Observed = train_scaled[[target]], 
#         Predicted = as.numeric(pred_train), 
#         Type = "Train"
#       ),
#       data.frame(
#         SampleID = rownames(test_scaled),  
#         Observed = test_scaled[[target]],  
#         Predicted = as.numeric(pred_test),  
#         Type = "OOD_Test"
#       )
#     )
#     pred_file <- paste0("fixed_", today, "_SVM_Linear_", tools::file_path_sans_ext(file), "_", target, "_pred.csv")
#     write.csv(df_pred_all, file.path(output_path, pred_file), row.names = FALSE)

#     # 6. 変数重要度
#     cat(sprintf("  Computing importance for %s...\n", target))
#     imps <- calc_permutation_importance(model, train_scaled, target, features, r2_train)
    
#     for (feat_name in names(imps)) {
#       imp_val <- imps[[feat_name]]
#       if (imp_val > 1e-5) {
#         feature_importance_data <- rbind(feature_importance_data, data.frame(
#           File = file, Target = target, Feature = feat_name, Importance_Mean = imp_val
#         ))
#       }
#     }
#     cat(sprintf("  Target: %s | R2: %.3f | OOD_R2: %.3f\n", target, r2_train, r2_ood))
#   }
# }

# # --- サマリー保存 ---
# sum_file <- file.path(output_path, paste0("fixed_", today, "_SVM_Linear_R_summary.csv"))
# write.csv(summary_data, sum_file, row.names = FALSE)
# imp_file <- file.path(output_path, paste0("fixed_", today, "_SVM_Linear_R_feature_importance.csv"))
# write.csv(feature_importance_data, imp_file, row.names = FALSE)

# cat(sprintf("\nSVM Linear Analysis Finished. Summary: %s\n", sum_file))

Created output directory: .//20251217_Rebuilt_12Files_Results
--- SVM Linear Analysis Started: 20251217 ---
Output: .//20251217_Rebuilt_12Files_Results

=== Processing SVM Linear: n_base.csv ===
  Computing importance for Jsc...
  Target: Jsc | R2: 0.935 | OOD_R2: 0.930
  Computing importance for Voc...
  Target: Voc | R2: 0.695 | OOD_R2: 0.886
  Computing importance for FF...
  Target: FF | R2: 0.656 | OOD_R2: 0.586
  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.949 | OOD_R2: 0.944

=== Processing SVM Linear: n_base_OH_rebuilt.csv ===


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: SMILESsnamenM_ICMA, SMILESsnamenM_TC61PM, SMILESsnamep1M_BDTH3, SMILESsnamep1M_BDTH8, SMILESsnamep1M_BDTH9, SMILESsnamep1M_CPDTDTKP, SMILESsnamep1M_PBDTTTST"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale 

  Computing importance for Jsc...
  Target: Jsc | R2: 0.983 | OOD_R2: 0.991


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: SMILESsnamep1M_BDTH3, SMILESsnamep1M_BDTH9, SMILESsnamep1M_CPDTDTKP, SMILESsnamep1M_PBTff4T2OD, SMILESsnamep1M_PNT4T2OD, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.2.3dioctylpyrazino.2.3d.pyridazine.P2, SMILESsnamep1M_Poly.N9.fheptadecanyl2.7carbazolealt5.5.4.f.7.fdi2thienyl.pyrazino.2.3d.pyridazine.P3"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant

  Computing importance for Voc...
  Target: Voc | R2: 0.764 | OOD_R2: 0.883


...

  Computing importance for Voc...
  Target: Voc | R2: 0.705 | OOD_R2: 0.471


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB, X111.3, X165.2, X165.5"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) 

  Computing importance for FF...
  Target: FF | R2: 0.737 | OOD_R2: 0.663


Warning message in preProcess.default(train_df[, feature_cols, drop = FALSE], method = "range"):
"No variation for for: THFSVAafetrN, THFSVAafetrI, cycle, Ratioip2M, Ratiop2M, Lay2Mname_ZnOC60, Lay5electronodes1_BCP, Lay5electronodes1_C60bis, Lay5electronodes1_Mg, Lay5electronodes1_TiOx, Additivesname_1.8dibromoooctane, Additivesname_1.8dicholorooctane, Additivesname_1.8dicyanooctane, Additivesname_1.8octanediacetate, Additivesname_1.8octanedithiol, Additivesname_CS2, Additivesname_DCB, Additivesname_DPE, namessolvent1_THF, namessolvent2_CB, namessolvent2_CF, namessolvent2_DIO, namessolvent2_TCB, X111.3, X165.2, X165.5"
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) `' constant. Cannot scale data."
Warning message in .local(x, ...):
"Variable(s) 

  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.972 | OOD_R2: 0.942

SVM Linear Analysis Finished. Summary: .//20251217_Rebuilt_12Files_Results/fixed_20251217_SVM_Linear_R_summary.csv


In [9]:
head(model$pred)

,C,pred,obs,rowIndex,Resample
,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,0.1,1.514765,1.39,349,Fold01
2,0.1,3.075424,3.60,308,Fold01
3,0.1,2.694717,2.80,340,Fold01
4,0.1,5.499848,5.50,386,Fold01
5,0.1,4.842859,4.69,328,Fold01
6,0.1,2.383189,2.34,542,Fold03
